# Step 3: Results Analysis

This notebook loads experiment results from Step 2 and produces:
- Summary statistics (mean ± standard deviation)
- Histograms comparing NAR vs NARX (RSS distributions)
- p-value and Cohen's d distribution plots

**Pipeline**: 01_hyperparameter_search → 02_run_experiment → `03_results_analysis`

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pickle
import matplotlib.pyplot as plt

from src.analysis import compute_mean_std, print_summary_table, plot_rss_histogram, plot_pvalue_cohensd

print("Imports successful.")

## Configuration

Specify the `.pkl` file from Step 2.

**Naming convention**: `results_{map}_{architecture}_{direction}_final.pkl`

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

PKL_FILE = "../results/experiments/results_henon_GRU_Y_to_X_final.pkl"

# Figure settings
FIG_WIDTH = 10
FIG_HEIGHT = 6
N_BINS = 30
SAVE_FIGURES = True
FIGURES_DIR = "../results/figures/"
SAVE_DPI = 300

# =============================================================================
print(f"Results file: {PKL_FILE}")
print(f"File exists: {os.path.exists(PKL_FILE)}")

## Load Results

In [ ]:
if not os.path.exists(PKL_FILE):
    raise FileNotFoundError(f"Results file not found: {PKL_FILE}")

with open(PKL_FILE, "rb") as f:
    results = pickle.load(f)

config = results.get("config", {})
chaotic_map = config.get("chaotic_map", "unknown")
direction = config.get("causality_direction", "unknown")
direction_str = config.get("direction_str", "")
target_var = config.get("target_variable", "")
cause_var = config.get("cause_variable", "")
architecture = config.get("nn_architecture", "unknown")

# Find lag key
lag_keys = [k for k in results if k.startswith("lag_")]
lag_key = lag_keys[0]
lag_data = results[lag_key]

# Detect RSS keys (supports old and new format)
if "RSS_restricted" in lag_data:
    key_nar, key_narx = "RSS_restricted", "RSS_full"
elif "RSS_X" in lag_data:
    key_nar, key_narx = "RSS_X", "RSS_XY"
elif "RSS_Y" in lag_data:
    key_nar, key_narx = "RSS_Y", "RSS_YX"
else:
    raise ValueError(f"Cannot detect RSS keys. Available: {list(lag_data.keys())}")

nar_label = f"NAR ({target_var} only)" if target_var else "NAR"
narx_label = f"NARX ({target_var} + {cause_var})" if target_var else "NARX"

print(f"Map: {chaotic_map} | Direction: {direction_str} | Arch: {architecture}")
print(f"Lag: {lag_key} | RSS keys: {key_nar}, {key_narx}")

## Summary Statistics (Mean ± Std)

In [ ]:
print_summary_table(
    lag_data, lag_key, chaotic_map, direction_str,
    architecture, key_nar, key_narx,
)

## Histogram: NAR vs NARX (RSS)

In [ ]:
rss_nar = np.asarray(lag_data.get(key_nar, []), dtype=float)
rss_narx = np.asarray(lag_data.get(key_narx, []), dtype=float)
rss_nar = rss_nar[~np.isnan(rss_nar)]
rss_narx = rss_narx[~np.isnan(rss_narx)]

if len(rss_nar) > 0 and len(rss_narx) > 0:
    save_path = None
    if SAVE_FIGURES:
        os.makedirs(FIGURES_DIR, exist_ok=True)
        save_path = os.path.join(
            FIGURES_DIR,
            f"histogram_{chaotic_map}_{architecture}_{direction}_{lag_key}.png"
        )

    fig = plot_rss_histogram(
        rss_nar, rss_narx, chaotic_map, direction, architecture, lag_key,
        nar_label=nar_label, narx_label=narx_label,
        fig_width=FIG_WIDTH, fig_height=FIG_HEIGHT, n_bins=N_BINS,
        save_path=save_path, dpi=SAVE_DPI,
    )
    plt.show()

    # Print comparison
    nar_m, nar_s, _ = compute_mean_std(rss_nar)
    narx_m, narx_s, _ = compute_mean_std(rss_narx)
    print(f"\n{key_nar} (NAR):  {nar_m:.4f} ± {nar_s:.4f}")
    print(f"{key_narx} (NARX): {narx_m:.4f} ± {narx_s:.4f}")
    if narx_m < nar_m:
        print(f"NARX reduces RSS by {(1 - narx_m / nar_m) * 100:.1f}%")
else:
    print("WARNING: Missing RSS data.")

## p-value and Cohen's d Distributions

In [ ]:
p_vals = np.asarray(lag_data.get("p_value", []), dtype=float)
d_vals = np.asarray(lag_data.get("cohens_d", []), dtype=float)
p_vals = p_vals[~np.isnan(p_vals)]
d_vals = d_vals[~np.isnan(d_vals)]

if len(p_vals) > 0 and len(d_vals) > 0:
    save_path2 = None
    if SAVE_FIGURES:
        save_path2 = os.path.join(
            FIGURES_DIR,
            f"histogram_stats_{chaotic_map}_{architecture}_{direction}_{lag_key}.png"
        )

    fig2 = plot_pvalue_cohensd(
        p_vals, d_vals, chaotic_map, direction, architecture, lag_key,
        save_path=save_path2, dpi=SAVE_DPI,
    )
    plt.show()
else:
    print("Not enough data for statistical distributions.")